## Import and Helper Functions

In [14]:
import requests
from bs4 import BeautifulSoup 
from html_to_markdown import convert_to_markdown
import pandas as pd 
import json

In [3]:
def stripStr(str):
    return ' '.join(c for c in str.split() if c)


def truncateIfLong(str, maxLen=45):
    if not isinstance(maxLen, int):
        maxLen = 15
    if maxLen < 4:
        maxLen = 5
    return str if len(str) < maxLen else str[:maxLen-3]+'...'


def miniStr(obj, lineSep=' ', wordSep=' ', trunc=None):
    mStr = lineSep.join(
        wordSep.join(w for w in line.split() if w)
        for line in str(obj).splitlines() if line.strip()
    )
    return truncateIfLong(mStr, trunc) if trunc else mStr


In [4]:
def reqStatus(r, isv=False, toRet=None):
    rs = f'<Response [{r.status_code} {r.reason}]> in {r.elapsed} from {r.url}'
    if isv:
        return (print(rs), r if toRet is None else toRet)[1]
    return rs

## Scrape Directory

In [11]:
def get_card_dets(card_soup, card_i=0, card_data=None, i_offset=0):
    i, a= card_i+i_offset, card_soup
    card_data = card_data.copy() if isinstance(card_data, dict) else {}
    card_data['card_no'] = i

    card_data['name'] = a.select_one('h3').get_text(' ', strip=True)
    card_data['title_grey'] = ' | '.join(
        [h.get_text(' ', strip=True) for h in a.select('h3+div h4')])
    
    p, c = 'total_posts_last30days', 'total_comments_last30days'
    card_data[p], card_data[c] = [
        int(s.get_text(' ', strip=True)) 
        for s in a.select('h3+div+div>div span')]

    card_data['link'] = (l:=a.get('href'))
    card_data['full_link'] = f"https://immilink.ca{l}".strip()
    return card_data

In [12]:
all_advisors, reqFn = [], requests.get
for pn in range(1,7):
    url = f'https://immilink.ca/advisors?page={pn}'
    soup = BeautifulSoup(reqStatus(reqFn(url),True).content,'html.parser')
    l = len(all_advisors)
    all_advisors += [
        get_card_dets(a,i,{'page_no':pn},l) for i, a in 
        enumerate(soup.select('a[href^="/R"]'), 1)]
print(f"Total {len(all_advisors)} advisors found")

<Response [200 OK]> in 0:00:03.188779 from https://immilink.ca/advisors?page=1
<Response [200 OK]> in 0:00:03.170718 from https://immilink.ca/advisors?page=2
<Response [200 OK]> in 0:00:02.988950 from https://immilink.ca/advisors?page=3
<Response [200 OK]> in 0:00:02.836670 from https://immilink.ca/advisors?page=4
<Response [200 OK]> in 0:00:02.967103 from https://immilink.ca/advisors?page=5
<Response [200 OK]> in 0:00:02.983176 from https://immilink.ca/advisors?page=6
Total 172 advisors found


### View Results

In [ ]:
# pd.DataFrame(all_advisors).to_csv('immilink_advisors.csv', index=False)
pd.DataFrame(all_advisors).set_index('card_no', drop=True)

,page_no,name,title_grey,total_posts_last30days,total_comments_last30days,link,full_link
card_no,,,,,,,
1,1,Neekee,RCIC-IRB | R1031440,6,0,/R1031440,https://immilink.ca/R1031440
2,1,Harpreet Singh,RCIC-IRB | R1036063,5,0,/R1036063,https://immilink.ca/R1036063
3,1,Andrei Zavialov,RCIC-IRB | R1034357,4,0,/R1034357,https://immilink.ca/R1034357
4,1,Narek,Paralegal (Ontario) | R1005184,3,0,/R1005184,https://immilink.ca/R1005184
5,1,Annu Malhi,RCIC-IRB | R713189,3,0,/R713189,https://immilink.ca/R713189
...,...,...,...,...,...,...,...
168,6,Nodirahon Boboeva,RCIC-IRB | R731024,0,0,/R731024,https://immilink.ca/R731024
169,6,Sheeba Joseph,RCIC-IRB | R712855,0,0,/R712855,https://immilink.ca/R712855
170,6,Mona Bakhtiari,RCIC-IRB | R713017,0,0,/R713017,https://immilink.ca/R713017


## Scrape Individual Advisors

In [17]:
def scrape_advisor(adv_card: dict) -> dict:
    al = adv_card['full_link'].strip()
    adv_data = {'card_no': adv_card['card_no'], 'immilink_page': al}

    print(adv_card['card_no'], 'Attempting to scrape', repr(al))
    rStat = reqStatus(resp:=requests.get(al))
    soup = BeautifulSoup(resp.content, 'html.parser')
    
    script_sel = 'script:-soup-contains-own("self.__next_f.push")'
    script_sel = f'{script_sel}:-soup-contains-own("advisorSlug")'
    try:
        s_txt = str(soup.select_one(script_sel)).replace('<script>','')
        s_txt = s_txt.replace('</script>','').strip()
        d = json.loads(json.loads('"'+s_txt.split(':',1)[-1].rstrip('])')))
        d = d[3]['children'][0][3]
        adv_data['advisorSlug'] = d['advisorSlug']
        adv_data['advisor_data'] = d['data']['advisor']
        adv_data['other_data'] = {
            k:v for k,v in d['data'].items() if k!='advisor'}
        
        am_span = soup.select_one('h3:-soup-contains-own("About Me")+span')
        adv_data['about_me_txt'] = '\n'.join([
            p.get_text(' ', strip=True) for p in am_span
        ]).strip()
        adv_data['about_me_md'] = convert_to_markdown(str(am_span))
        adv_data['about_me_html'] = str(am_span.prettify())
    except Exception as e:
        adv_data['scrape_error'] = f'<{type(e)}> {e!r}'

    adv_data['http_status'] = rStat
    return adv_data

In [18]:
all_adv_data = [scrape_advisor(a) for a in all_advisors]

1 Attempting to scrape 'https://immilink.ca/R1031440'
2 Attempting to scrape 'https://immilink.ca/R1036063'
3 Attempting to scrape 'https://immilink.ca/R1034357'
4 Attempting to scrape 'https://immilink.ca/R1005184'
5 Attempting to scrape 'https://immilink.ca/R713189'
6 Attempting to scrape 'https://immilink.ca/R713190'
7 Attempting to scrape 'https://immilink.ca/R731277'
8 Attempting to scrape 'https://immilink.ca/R713326'
9 Attempting to scrape 'https://immilink.ca/R713017'
10 Attempting to scrape 'https://immilink.ca/R731024'
11 Attempting to scrape 'https://immilink.ca/R731480'
12 Attempting to scrape 'https://immilink.ca/R713218'
13 Attempting to scrape 'https://immilink.ca/R731331'
14 Attempting to scrape 'https://immilink.ca/R712670'
15 Attempting to scrape 'https://immilink.ca/R731011'
16 Attempting to scrape 'https://immilink.ca/R712855'
17 Attempting to scrape 'https://immilink.ca/R508428'
18 Attempting to scrape 'https://immilink.ca/R713020'
19 Attempting to scrape 'https://

In [19]:
with open('all_adv_data.json', 'w') as f:
    json.dump(all_adv_data, f, indent=4)

## Next

# END